## M4 — Clustering K-Means
Segmentamos las 100 ubicaciones en perfiles para enriquecer la narrativa ante el jurado.

In [8]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
import warnings
warnings.filterwarnings('ignore')

df = pd.read_csv('top100_ubicaciones.csv', encoding='utf-8-sig')

print(f"Ubicaciones cargadas: {df.shape[0]}")
print(f"Columnas: {df.columns.tolist()}")

Ubicaciones cargadas: 100
Columnas: ['CPRO', 'CMUN', 'NOMBRE', 'secciones', 'fofm_medio', 'fofm_max', 'vol_75plus_total', 'vol_80plus_total', 'pct_75plus_media', 'ratio_viudez_media', 'cohorte_futura', 'tasa_sustitucion', 'renta_hogar', 'pension_media', 'precio_m2', 'indice_cobertura', 'tiene_hospital', 'latitud', 'longitud', 'ranking_municipio', 'posicion_final']


### Paso 1 — Seleccionar variables y escalar

In [9]:
# Variables que definen el perfil de cada municipio
features = [
    'renta_hogar',
    'vol_75plus_total',
    'pct_75plus_media',
    'ratio_viudez_media',
    'cohorte_futura'
]

X = df[features].copy()

# Escalar — media 0, desviación estándar 1
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print("Variables escaladas:")
print(pd.DataFrame(X_scaled, columns=features).describe().round(2))

Variables escaladas:
       renta_hogar  vol_75plus_total  pct_75plus_media  ratio_viudez_media  \
count       100.00            100.00            100.00              100.00   
mean         -0.00              0.00              0.00                0.00   
std           1.01              1.01              1.01                1.01   
min          -0.79             -0.58             -2.48               -2.41   
25%          -0.51             -0.51             -0.62               -0.60   
50%          -0.23             -0.34              0.00                0.01   
75%           0.11              0.10              0.63                0.65   
max           6.40              7.37              2.43                4.68   

       cohorte_futura  
count          100.00  
mean             0.00  
std              1.01  
min             -0.64  
25%             -0.55  
50%             -0.33  
75%              0.09  
max              6.80  


### Paso 2 — Encontrar el número óptimo de clusters

In [10]:
resultados = []

for k in range(2, 9):
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = kmeans.fit_predict(X_scaled)
    sil    = silhouette_score(X_scaled, labels)
    inercia = kmeans.inertia_
    resultados.append({'k': k, 'silhouette': round(sil, 4), 'inercia': round(inercia, 1)})

df_res = pd.DataFrame(resultados)
print(df_res.to_string(index=False))

 k  silhouette  inercia
 2      0.2985    378.4
 3      0.2399    292.2
 4      0.3015    216.2
 5      0.3123    182.9
 6      0.2532    150.3
 7      0.2738    125.7
 8      0.2597    109.8


### Paso 3 — Aplicar K-Means con 4 clusters

In [11]:
kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
df['cluster'] = kmeans.fit_predict(X_scaled)

# Ver cuántos municipios hay en cada cluster
print("Municipios por cluster:")
print(df['cluster'].value_counts().sort_index())

# Ver las características medias de cada cluster
perfil = df.groupby('cluster')[features].mean().round(0)
print("\nPerfil medio por cluster:")
print(perfil.to_string())

Municipios por cluster:
cluster
0    55
1    37
2     4
3     4
Name: count, dtype: int64

Perfil medio por cluster:
         renta_hogar  vol_75plus_total  pct_75plus_media  ratio_viudez_media  cohorte_futura
cluster                                                                                     
0            29910.0            5158.0               9.0                 2.0          6098.0
1            30502.0           19065.0              12.0                 2.0         20254.0
2            66070.0            4997.0               8.0                 2.0          6564.0
3            32320.0          103064.0              11.0                 2.0        103194.0


### Paso 4 — Etiquetar los clusters con nombres estratégicos

In [12]:
nombres_cluster = {
    0: "Zona envejecida — solvencia ajustada",
    1: "Ciudad media — mercado relevante",
    2: "Zona premium — máxima solvencia",
    3: "Gran ciudad — mercado masivo"
}

df['perfil'] = df['cluster'].map(nombres_cluster)

resumen = df.groupby('perfil').agg(
    municipios      = ('NOMBRE', 'count'),
    renta_media     = ('renta_hogar', 'mean'),
    vol_75plus      = ('vol_75plus_total', 'mean'),
    fofm_medio      = ('fofm_medio', 'mean')
).round(2)

print("=== PERFILES ESTRATÉGICOS ACTUALIZADOS ===")
print(resumen.to_string())

for perfil in nombres_cluster.values():
    municipios = df[df['perfil'] == perfil]['NOMBRE'].tolist()
    print(f"\n{perfil} ({len(municipios)}):")
    print(", ".join(municipios[:8]))
    if len(municipios) > 8:
        print(f"  ... y {len(municipios)-8} más")

=== PERFILES ESTRATÉGICOS ACTUALIZADOS ===
                                      municipios  renta_media  vol_75plus  fofm_medio
perfil                                                                               
Ciudad media — mercado relevante              37     30502.38    19064.73        0.28
Gran ciudad — mercado masivo                   4     32319.75   103063.75        0.35
Zona envejecida — solvencia ajustada          55     29910.47     5158.04        0.27
Zona premium — máxima solvencia                4     66069.75     4997.25        0.51

Zona envejecida — solvencia ajustada (55):
Sant Esteve Sesrovires, Galdakao, Sagunto/Sagunt, Sant Joan d'Alacant, Hondarribia, Eivissa, Marbella, Alzira
  ... y 47 más

Ciudad media — mercado relevante (37):
Bilbao, Donostia/San Sebastián, Cádiz, Tolosa, Alicante/Alacant, Pamplona/Iruña, Barakaldo, Málaga
  ... y 29 más

Zona premium — máxima solvencia (4):
Pozuelo de Alarcón, Majadahonda, Torrelodones, Sant Cugat del Vallès

Gran ciuda

### Paso 5 — Guardar resultados del clustering

In [13]:
# Guardar top100 con cluster asignado
df.to_csv('top100_con_clusters.csv', index=False, encoding='utf-8-sig')

# Excel actualizado con la columna de perfil
df_muni = pd.read_csv('ranking_municipios.csv', encoding='utf-8-sig')
resumen_prov = pd.read_csv('resumen_provincias.csv', encoding='utf-8-sig')

with pd.ExcelWriter('resultados_akademia.xlsx', engine='openpyxl') as writer:
    df.to_excel(writer, sheet_name='Top 100 ubicaciones', index=False)
    df_muni.head(500).to_excel(writer, sheet_name='Ranking municipios', index=False)
    resumen_prov.to_excel(writer, sheet_name='Resumen por provincia', index=False)
    
    # Nueva pestaña con resumen de perfiles
    resumen_clusters = df.groupby('perfil').agg(
        municipios  = ('NOMBRE', 'count'),
        renta_media = ('renta_hogar', 'mean'),
        vol_75plus  = ('vol_75plus_total', 'mean'),
        fofm_medio  = ('fofm_medio', 'mean')
    ).round(2).reset_index()
    resumen_clusters.to_excel(writer, sheet_name='Perfiles estratégicos', index=False)

print("Archivos guardados:")
print(f"  top100_con_clusters.csv → {len(df)} municipios con perfil asignado")
print(f"  resultados_akademia.xlsx → actualizado con pestaña de perfiles")

Archivos guardados:
  top100_con_clusters.csv → 100 municipios con perfil asignado
  resultados_akademia.xlsx → actualizado con pestaña de perfiles
